In [4]:
import ipywidgets as widgets
from IPython.display import display

# Create the upload button
uploader = widgets.FileUpload(multiple=True)
display(uploader)

FileUpload(value={}, description='Upload', multiple=True)

In [5]:
# Run this in a separate cell after uploading to save the files locally in the runtime
if uploader.value:
    for filename, file_info in uploader.value.items():
        with open(filename, "wb") as f:
            f.write(file_info['content'])
    print("Upload successful!")

In [6]:
%pip install -U python-dotenv pymongo 'transformers>=5.4' peft scikit-learn pandas numpy tqdm lime shap joblib parsel

  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 48.6 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1


In [7]:
from __future__ import annotations

import argparse
import json
import os
import re
from collections import Counter
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from lime.lime_text import LimeTextExplainer
from pymongo import MongoClient
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

from extract_inputs_jsonl import (
    collection_possible_labels,
    iter_documents,
    normalize_document,
    record_label_for_source,
    selected_collections,
)

In [8]:
# Configuration
PROJECT_ROOT = Path(".")
ENV_FILE = PROJECT_ROOT / ".env"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "mongo_embedding_lime_shap"

MONGO_SOURCE = "all"  # phishing | benign | urlscan_live | all
LIMIT_PER_LABEL = 10000  # 0 means no cap
LIMIT_PER_SOURCE = 10000  # 0 means no cap
INCLUDE_HTTP_ERRORS = False
INCLUDE_GENERIC_ERROR_PAGES = False
BATCH_SIZE = 500
RANDOM_SEED = 3407

EMBEDDING_MODEL_NAME = "jinaai/jina-embeddings-v3-hf"
EMBEDDING_TASK = "classification"
EMBED_BATCH_SIZE = 16
EMBED_MAX_TOKENS = 2048
MAX_HTML_CHARS = 12000
MAX_REDIRECTS = 10

TEST_SIZE = 0.20
CLASSIFIER_MAX_ITER = 2500

LIME_SAMPLE_COUNT = 5
LIME_NUM_FEATURES = 15

SHAP_SAMPLE_COUNT = 3
SHAP_MAX_EVALS = 500
SHAP_TOP_K = 20

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Artifacts:", OUTPUT_DIR.resolve())
print("Mongo source:", MONGO_SOURCE)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding task:", EMBEDDING_TASK)
print("Limit per source:", LIMIT_PER_SOURCE)

Artifacts: /content/artifacts/mongo_embedding_lime_shap
Mongo source: all
Embedding model: jinaai/jina-embeddings-v3-hf
Embedding task: classification
Limit per source: 10000


In [9]:
def collapse_ws(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def truncate_keep_ends(text: str, max_chars: int) -> str:
    text = collapse_ws(text)
    if len(text) <= max_chars:
        return text

    head_chars = int(max_chars * 0.7)
    tail_chars = max_chars - head_chars
    head = text[:head_chars].rstrip()
    tail = text[-tail_chars:].lstrip()
    return f"{head}\n<!-- HTML_TRUNCATED -->\n{tail}"


def extract_redirects_from_metadata(metadata: dict[str, Any]) -> list[dict[str, Any]]:
    redirects: list[dict[str, Any]] = []
    for redirect in metadata.get("redirect_history") or []:
        if not isinstance(redirect, dict):
            continue
        item: dict[str, Any] = {}
        status_code = redirect.get("status_code")
        url = collapse_ws(redirect.get("url"))
        if status_code is not None:
            item["status_code"] = status_code
        if url:
            item["url"] = url
        if item:
            redirects.append(item)
    return redirects


def render_redirects(redirects: list[dict[str, Any]]) -> str:
    if not redirects:
        return "none"

    lines: list[str] = []
    for index, redirect in enumerate(redirects[:MAX_REDIRECTS], start=1):
        status_code = redirect.get("status_code", "unknown")
        url = collapse_ws(redirect.get("url"))
        if url:
            lines.append(f"{index}. status={status_code} url={url}")
        else:
            lines.append(f"{index}. status={status_code}")
    return "\n".join(lines)


def build_embedding_text(normalized_doc: dict[str, Any]) -> str:
    metadata = normalized_doc.get("metadata") or {}
    url = collapse_ws(normalized_doc.get("url") or metadata.get("url"))
    final_url = collapse_ws(metadata.get("final_url") or url)
    redirects = extract_redirects_from_metadata(metadata)
    raw_html = truncate_keep_ends(normalized_doc.get("html") or "", MAX_HTML_CHARS)

    return "\n\n".join(
        [
            "[URL]",
            url or "missing",
            "[FINAL_URL]",
            final_url or "missing",
            "[REDIRECTS]",
            render_redirects(redirects),
            "[RAW_HTML]",
            raw_html or "missing",
        ]
    )


def apply_caps(
    df: pd.DataFrame,
    limit_per_label: int = 0,
    limit_per_source: int = 0,
) -> pd.DataFrame:
    if df.empty or (not limit_per_label and not limit_per_source):
        return df

    kept_rows: list[dict[str, Any]] = []
    label_counts: Counter[str] = Counter()
    source_counts: Counter[str] = Counter()

    for row in df.to_dict(orient="records"):
        label = str(row["label"])
        source = str(row["source"])
        if limit_per_label and label_counts[label] >= limit_per_label:
            continue
        if limit_per_source and source_counts[source] >= limit_per_source:
            continue

        kept_rows.append(row)
        label_counts[label] += 1
        source_counts[source] += 1

    capped_df = pd.DataFrame(kept_rows)
    print("Final capped label counts:", label_counts)
    print("Final capped source counts:", source_counts)
    return capped_df


def source_cap_reached(
    source_name: str,
    possible_labels: set[str],
    label_counts: Counter[str],
    source_counts: Counter[str],
    limit_per_label: int = 0,
    limit_per_source: int = 0,
) -> bool:
    if limit_per_source and source_counts[source_name] >= limit_per_source:
        return True
    if limit_per_label and possible_labels and all(label_counts[label] >= limit_per_label for label in possible_labels):
        return True
    return False


def make_extraction_args() -> argparse.Namespace:
    return argparse.Namespace(
        include_http_errors=INCLUDE_HTTP_ERRORS,
        include_generic_error_pages=INCLUDE_GENERIC_ERROR_PAGES,
        batch_size=BATCH_SIZE,
        no_progress=False,
    )


def load_records_from_mongo(
    source: str,
    limit_per_label: int = 0,
    limit_per_source: int = 0,
) -> pd.DataFrame:
    load_dotenv(ENV_FILE)
    mongo_uri = os.getenv("MONGO_URI")
    if not mongo_uri:
        raise RuntimeError(f"MONGO_URI is not set. Expected it in {ENV_FILE.resolve()}.")

    args = make_extraction_args()
    rows: list[dict[str, Any]] = []
    label_counts: Counter[str] = Counter()
    source_counts: Counter[str] = Counter()
    target_labels = {"phishing", "benign"}

    with MongoClient(mongo_uri) as client:
        for label_name, db_name, collection_name in selected_collections(source):
            source_name = f"{db_name}.{collection_name}"
            possible_labels = collection_possible_labels(label_name) & target_labels
            if source_cap_reached(
                source_name,
                possible_labels,
                label_counts,
                source_counts,
                limit_per_label=limit_per_label,
                limit_per_source=limit_per_source,
            ):
                print(f"Skipping {source_name}: caps already satisfied.")
                continue
            print(f"Loading {source_name}...")

            for doc in iter_documents(client, db_name, collection_name, args):
                if source_cap_reached(
                    source_name,
                    possible_labels,
                    label_counts,
                    source_counts,
                    limit_per_label=limit_per_label,
                    limit_per_source=limit_per_source,
                ):
                    break
                label = record_label_for_source(label_name, doc)
                if label not in target_labels:
                    continue
                if limit_per_label and label_counts[label] >= limit_per_label:
                    continue

                normalized_doc = normalize_document(doc)
                embedding_text = build_embedding_text(normalized_doc)
                metadata = normalized_doc.get("metadata") or {}
                redirects = extract_redirects_from_metadata(metadata)

                rows.append(
                    {
                        "id": str(normalized_doc.get("_id") or doc.get("_id") or len(rows)),
                        "label": label,
                        "source": source_name,
                        "url": collapse_ws(normalized_doc.get("url") or metadata.get("url")),
                        "final_url": collapse_ws(metadata.get("final_url")),
                        "redirect_count": len(redirects),
                        "html_chars": len(normalized_doc.get("html") or ""),
                        "text": embedding_text,
                    }
                )
                label_counts[label] += 1
                source_counts[source_name] += 1

                if limit_per_label and all(label_counts[name] >= limit_per_label for name in target_labels):
                    break

    if not rows:
        raise RuntimeError("No records were loaded from MongoDB after filtering.")

    df = pd.DataFrame(rows)
    print("Loaded rows before final cap pass:", len(df))
    print("Streaming label counts:", label_counts)
    print("Streaming source counts:", source_counts)
    df = apply_caps(
        df,
        limit_per_label=limit_per_label,
        limit_per_source=limit_per_source,
    )
    print("Loaded rows after final cap pass:", len(df))
    return df


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode_texts(
    model: AutoModel,
    tokenizer: AutoTokenizer,
    texts: list[str],
    batch_size: int = EMBED_BATCH_SIZE,
    max_length: int = EMBED_MAX_TOKENS,
    show_progress_bar: bool = False,
) -> np.ndarray:
    vectors: list[np.ndarray] = []
    iterator = range(0, len(texts), batch_size)
    if show_progress_bar:
        iterator = tqdm(iterator, desc="Embedding", leave=False)

    model.eval()
    with torch.no_grad():
        for start in iterator:
            batch_texts = texts[start:start + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {key: value.to(model.device) for key, value in encoded.items()}
            outputs = model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            pooled = F.normalize(pooled, p=2, dim=1)
            vectors.append(pooled.cpu().numpy().astype(np.float32))

    return np.concatenate(vectors, axis=0)


def save_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def safe_stem(value: str) -> str:
    slug = re.sub(r"[^a-zA-Z0-9._-]+", "_", value).strip("._")
    return slug or "sample"


def summarise_shap_values(
    shap_values: Any,
    sample_index: int,
    class_index: int,
    top_k: int,
) -> list[dict[str, Any]]:
    tokens = [str(token) for token in shap_values.data[sample_index]]
    values = np.asarray(shap_values.values[sample_index])
    if values.ndim == 1:
        token_scores = values
    else:
        token_scores = values[:, class_index]

    ranked = sorted(
        zip(tokens, token_scores),
        key=lambda item: abs(float(item[1])),
        reverse=True,
    )
    summary: list[dict[str, Any]] = []
    for token, score in ranked[:top_k]:
        cleaned = collapse_ws(token)
        if not cleaned:
            continue
        summary.append(
            {
                "token": cleaned,
                "score": float(score),
                "direction": "supports" if score >= 0 else "opposes",
            }
        )
    return summary




In [10]:
# Load records directly from MongoDB.
df = load_records_from_mongo(
    MONGO_SOURCE,
    limit_per_label=LIMIT_PER_LABEL,
    limit_per_source=LIMIT_PER_SOURCE,
)

dataset_preview_path = OUTPUT_DIR / "dataset_preview.json"
save_json(
    dataset_preview_path,
    {
        "rows": int(len(df)),
        "limit_per_label": LIMIT_PER_LABEL,
        "limit_per_source": LIMIT_PER_SOURCE,
        "label_counts": df["label"].value_counts().to_dict(),
        "sources": df["source"].value_counts().to_dict(),
        "preview": df.head(3).to_dict(orient="records"),
    },
)
print("Saved preview:", dataset_preview_path.resolve())




Loading phishing_db.website_content...


phishing_db.website_content: 13370doc [01:39, 134.65doc/s]


Loading tranco.websites...


tranco.websites: 10619doc [08:25, 21.02doc/s]


Skipping urlscan.live: caps already satisfied.
Loaded rows before final cap pass: 20000
Streaming label counts: Counter({'phishing': 10000, 'benign': 10000})
Streaming source counts: Counter({'phishing_db.website_content': 10000, 'tranco.websites': 10000})
Final capped label counts: Counter({'phishing': 10000, 'benign': 10000})
Final capped source counts: Counter({'phishing_db.website_content': 10000, 'tranco.websites': 10000})
Loaded rows after final cap pass: 20000
Saved preview: /content/artifacts/mongo_embedding_lime_shap/dataset_preview.json


In [11]:
# Train / test split
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["label"])
class_names = [str(name) for name in label_encoder.classes_]

train_df, test_df, y_train, y_test = train_test_split(
    df,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y,
)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Classes:", class_names)




Train rows: 16000
Test rows: 4000
Classes: ['benign', 'phishing']


In [12]:
# Embeddings + classifier
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
embedder = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME)
embedder.load_adapter(
    EMBEDDING_MODEL_NAME,
    adapter_name=EMBEDDING_TASK,
    adapter_kwargs={"subfolder": EMBEDDING_TASK},
)
embedder.set_adapter(EMBEDDING_TASK)
if torch.cuda.is_available():
    embedder = embedder.to("cuda")
print("Embedding device:", embedder.device)

X_train = encode_texts(
    embedder,
    tokenizer,
    train_df["text"].tolist(),
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
)
X_test = encode_texts(
    embedder,
    tokenizer,
    test_df["text"].tolist(),
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
)

classifier = LogisticRegression(
    max_iter=CLASSIFIER_MAX_ITER,
    class_weight="balanced",
    random_state=RANDOM_SEED,
)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
y_prob = classifier.predict_proba(X_test)

report = classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(y_test, y_pred).tolist()

metrics_payload = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "embedding_task": EMBEDDING_TASK,
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "class_names": class_names,
    "classification_report": report,
    "confusion_matrix": cm,
}
metrics_path = OUTPUT_DIR / "metrics.json"
save_json(metrics_path, metrics_payload)
print("Saved metrics:", metrics_path.resolve())




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/294 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
# Persist the trained classifier and metadata.
model_bundle_path = OUTPUT_DIR / "embedding_classifier.joblib"
joblib.dump(
    {
        "classifier": classifier,
        "label_encoder": label_encoder,
        "embedding_model_name": EMBEDDING_MODEL_NAME,
        "embedding_task": EMBEDDING_TASK,
        "max_html_chars": MAX_HTML_CHARS,
        "max_redirects": MAX_REDIRECTS,
    },
    model_bundle_path,
)
print("Saved model bundle:", model_bundle_path.resolve())

In [ ]:
# Predict function used by LIME and SHAP.
def predict_proba_from_texts(texts: list[str] | np.ndarray) -> np.ndarray:
    text_list = [str(text) for text in texts]
    embedded = encode_texts(
        embedder,
        tokenizer,
        text_list,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=False,
    )
    return classifier.predict_proba(embedded)


test_with_predictions = test_df.copy()
test_with_predictions["y_true"] = y_test
test_with_predictions["y_pred"] = y_pred
test_with_predictions["predicted_label"] = label_encoder.inverse_transform(y_pred)
test_with_predictions["true_label"] = label_encoder.inverse_transform(y_test)
test_with_predictions["predicted_confidence"] = y_prob.max(axis=1)
test_with_predictions = test_with_predictions.sort_values(
    by="predicted_confidence",
    ascending=False,
)

In [ ]:
# LIME explanations on a few high-confidence test samples.
lime_dir = OUTPUT_DIR / "lime"
lime_dir.mkdir(parents=True, exist_ok=True)

lime_explainer = LimeTextExplainer(class_names=class_names)
lime_records: list[dict[str, Any]] = []

lime_samples = test_with_predictions.head(LIME_SAMPLE_COUNT)
for row in tqdm(
    list(lime_samples.itertuples(index=False)),
    desc="LIME",
):
    predicted_index = int(row.y_pred)
    explanation = lime_explainer.explain_instance(
        row.text,
        classifier_fn=predict_proba_from_texts,
        num_features=LIME_NUM_FEATURES,
        labels=[predicted_index],
    )

    sample_name = safe_stem(f"{row.id}_{row.predicted_label}")
    html_path = lime_dir / f"{sample_name}.html"
    json_path = lime_dir / f"{sample_name}.json"
    explanation.save_to_file(str(html_path), labels=[predicted_index])

    payload = {
        "id": row.id,
        "source": row.source,
        "true_label": row.true_label,
        "predicted_label": row.predicted_label,
        "predicted_confidence": float(row.predicted_confidence),
        "features": [
            {"token": token, "weight": float(weight)}
            for token, weight in explanation.as_list(label=predicted_index)
        ],
    }
    save_json(json_path, payload)
    lime_records.append(payload)

save_json(OUTPUT_DIR / "lime_summary.json", lime_records)
print("Saved LIME explanations:", lime_dir.resolve())




In [ ]:
# SHAP text explanations against the same embedding-backed predictor.
shap_dir = OUTPUT_DIR / "shap"
shap_dir.mkdir(parents=True, exist_ok=True)

shap_masker = shap.maskers.Text()
shap_explainer = shap.Explainer(
    predict_proba_from_texts,
    shap_masker,
    output_names=class_names,
)

shap_records: list[dict[str, Any]] = []
shap_samples = test_with_predictions.head(SHAP_SAMPLE_COUNT)
shap_inputs = shap_samples["text"].tolist()
shap_values = shap_explainer(shap_inputs, max_evals=SHAP_MAX_EVALS)

for sample_index, row in enumerate(shap_samples.itertuples(index=False)):
    predicted_index = int(row.y_pred)
    payload = {
        "id": row.id,
        "source": row.source,
        "true_label": row.true_label,
        "predicted_label": row.predicted_label,
        "predicted_confidence": float(row.predicted_confidence),
        "top_tokens": summarise_shap_values(
            shap_values=shap_values,
            sample_index=sample_index,
            class_index=predicted_index,
            top_k=SHAP_TOP_K,
        ),
    }
    sample_name = safe_stem(f"{row.id}_{row.predicted_label}")
    save_json(shap_dir / f"{sample_name}.json", payload)
    shap_records.append(payload)

save_json(OUTPUT_DIR / "shap_summary.json", shap_records)
print("Saved SHAP explanations:", shap_dir.resolve())

In [ ]:
# Save the test predictions for later inspection.
predictions_path = OUTPUT_DIR / "test_predictions.csv"
test_with_predictions.drop(columns=["text"]).to_csv(predictions_path, index=False)
print("Saved test predictions:", predictions_path.resolve())

In [ ]:
def score_single_document(doc: dict[str, Any]) -> dict[str, Any]:
    normalized_doc = normalize_document(doc)
    text = build_embedding_text(normalized_doc)
    probabilities = predict_proba_from_texts([text])[0]
    best_index = int(np.argmax(probabilities))
    return {
        "predicted_label": class_names[best_index],
        "confidence": float(probabilities[best_index]),
        "probabilities": {
            class_name: float(probabilities[index])
            for index, class_name in enumerate(class_names)
        },
    }